# Проверка среды и доступных моделей
Этот блокнот проверяет доступ к модели, а не качество агента. Выполните ячейки сверху вниз. По умолчанию mock работает без внешних запросов. Для практики с реализацией функций откройте `01-agent-student.ipynb`.

## 1. Подключение модели · 10 минут
Сначала оставьте `BACKEND = "mock"`: этот режим позволяет выполнить все обязательные задачи без ключа и загрузки весов. Live-эксперимент в конце выполняется при доступной инфраструктуре; отсутствие API/GPU не снижает оценку за код.

Режимы: `mock` — заранее подготовленные ответы для проверки Python; `local` — веса в вычислительной среде Colab; `openrouter` — запрос внешнему провайдеру. Вне Colab `local` использует ту машину, где работает kernel. Не отправляйте секреты и личные данные.

Для local в Colab включите GPU: «Среда выполнения → Сменить среду выполнения». GPU может быть недоступен. Qwen3-0.6B — небольшая учебная модель, её качество не гарантирует решение задачи.

Для OpenRouter создайте ключ и сохраните его в Colab Secrets под именем `OPENROUTER_API_KEY`, разрешив этому блокноту доступ к секрету. Не вставляйте ключ в код. Сначала выполните конфигурацию и каталог, затем скопируйте выбранный `:free` id в `MODEL_ID`, повторите конфигурацию и создайте клиент. Не нужно повторно выполнять установку зависимостей.

Если настройка занимает больше 10 минут, переходите к задачам в mock; диагностику live можно завершить отдельно.

In [1]:
BACKEND = "mock"  # mock | local | openrouter
MODEL_ID = None  # Для OpenRouter скопируйте id из каталога ниже
MAX_CALLS = 12

In [2]:
if BACKEND == "local":
    %pip -q install "transformers==4.56.2" "accelerate==1.10.1"
# torch уже установлен в обычной среде Colab; локально установите его отдельно.

### Готовый модуль
Эта свёрнутая ячейка — инфраструктура, её не нужно переписывать. Она ограничивает число вызовов, читает ключ без сохранения в notebook и допускает только модели, прошедшие проверку бесплатного каталога. Ограничение вызовов действует на текущий экземпляр клиента; это учебный бюджет, а не защита от намеренного обхода.

In [3]:
"""Учебный доступ к LLM: стандартная библиотека для OpenRouter, lazy local backend."""
import json
import os
import time
from decimal import Decimal, InvalidOperation
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

BASE_URL = 'https://openrouter.ai/api/v1'

class ModelAccessError(RuntimeError):
    pass

def _request(path, key=None, payload=None):
    headers = {'Content-Type': 'application/json'}
    if key:
        headers['Authorization'] = 'Bearer ' + key
    req = Request(BASE_URL + path, headers=headers,
                  data=None if payload is None else json.dumps(payload).encode())
    try:
        with urlopen(req, timeout=45) as response:
            result = json.load(response)
    except (json.JSONDecodeError, UnicodeDecodeError):
        raise ModelAccessError('API вернул некорректный JSON. Повторите позже; ответ не выводится.') from None
    except HTTPError as exc:
        hints = {401: 'Проверьте API-ключ.', 402: 'Запрос отклонён по балансу; платного fallback нет.',
                 403: 'Проверьте доступ и настройки аккаунта.',
                 429: 'Лимит запросов; подождите или используйте local/mock.'}
        raise ModelAccessError(f'HTTP {exc.code}. ' + hints.get(exc.code, 'Ошибка провайдера; повторите позже.')) from None
    except (URLError, TimeoutError, OSError):
        raise ModelAccessError('Сеть недоступна или истёк таймаут.') from None
    if not isinstance(result, dict):
        raise ModelAccessError('API вернул JSON неожиданного формата: ожидается объект.')
    if 'error' in result:
        raise ModelAccessError('API вернул ошибку. Ответ и ключ не выводятся.')
    return result

def is_free_text_model(model):
    """Консервативный фильтр: :free и нулевые опубликованные цены."""
    if not isinstance(model, dict):
        return False
    pricing = model.get('pricing') or {}
    model_id = model.get('id')
    if not isinstance(model_id, str) or not model_id.endswith(':free'):
        return False
    if not isinstance(pricing, dict) or not all(k in pricing for k in ('prompt', 'completion')):
        return False
    try:
        if any(Decimal(str(value)) != 0 for value in pricing.values()):
            return False
    except (InvalidOperation, ValueError, TypeError):
        return False
    arch = model.get('architecture') or {}
    if not isinstance(arch, dict):
        return False
    if not all(isinstance(arch.get(k), list) for k in ('input_modalities', 'output_modalities')):
        return False
    return 'text' in arch.get('input_modalities', []) and 'text' in arch.get('output_modalities', [])

def free_models(require_tools=False):
    data = _request('/models').get('data')
    if not isinstance(data, list):
        raise ModelAccessError('Каталог моделей имеет неожиданный формат. Повторите позже.')
    return sorted([m for m in data if is_free_text_model(m)
                   and (not require_tools or 'tools' in (m.get('supported_parameters') or []))],
                  key=lambda m: m['id'])

def read_key():
    key = os.environ.get('OPENROUTER_API_KEY')
    if not key:
        try:
            from google.colab import userdata
            key = userdata.get('OPENROUTER_API_KEY')
        except Exception:
            pass
    if not key:
        from getpass import getpass
        key = getpass('OpenRouter API key (скрытый ввод): ').strip()
    if not key:
        raise ModelAccessError('API-ключ не задан.')
    return key

class ModelClient:
    def __init__(self, backend='mock', model_id=None, key=None, max_calls=12):
        if backend not in ('mock', 'openrouter', 'local'):
            raise ValueError('backend: mock, openrouter или local')
        self.backend, self.model_id, self.key = backend, model_id, key
        self.max_calls, self.calls = max_calls, 0
        self.records = []
        self.tokenizer = self.model = None
        if backend == 'openrouter':
            if model_id not in {m['id'] for m in free_models()}:
                raise ModelAccessError('Выберите модель из текущего бесплатного каталога.')
            self.key = key or read_key()
        if backend == 'local':
            self.model_id = model_id or 'Qwen/Qwen3-0.6B'

    def _load_local(self):
        try:
            import torch
            from transformers import AutoTokenizer, AutoModelForCausalLM
        except ImportError:
            raise ModelAccessError('Для local нужны torch и transformers. Выполните ячейку установки.') from None
        try:
            tokenizer = AutoTokenizer.from_pretrained(self.model_id)
            model = AutoModelForCausalLM.from_pretrained(
                self.model_id, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
            model.to('cuda' if torch.cuda.is_available() else 'cpu').eval()
        except OSError:
            raise ModelAccessError('Не удалось загрузить локальную модель. Проверьте сеть и model_id.') from None
        self.tokenizer, self.model = tokenizer, model

    def chat(self, messages, max_tokens=128):
        if self.calls >= self.max_calls:
            raise ModelAccessError('Исчерпан учебный бюджет вызовов. Не перезапускайте ячейку ради обхода лимитов провайдера.')
        if not 1 <= max_tokens <= 512:
            raise ValueError('Учебный лимит: 1–512 выходных токенов.')
        self.calls += 1
        started = time.monotonic()
        status = 'error'
        try:
            if self.backend == 'mock':
                answer = '[MOCK: это заглушка, не ответ LLM]'
            elif self.backend == 'openrouter':
                # Повторная проверка каталога перед каждым вызовом; без auto-router и fallback.
                if self.model_id not in {m['id'] for m in free_models()}:
                    raise ModelAccessError('Модель больше не прошла проверку бесплатности.')
                result = _request('/chat/completions', self.key, {
                    'model': self.model_id, 'messages': messages, 'max_tokens': max_tokens,
                    'provider': {'allow_fallbacks': False}})
                choices = result.get('choices')
                if not isinstance(choices, list) or not choices or not isinstance(choices[0], dict):
                    raise ModelAccessError('API вернул ответ без списка choices. Повторите позже.')
                message = choices[0].get('message')
                if not isinstance(message, dict):
                    raise ModelAccessError('API вернул ответ без объекта message. Повторите позже.')
                answer = message.get('content')
            else:
                if self.model is None:
                    self._load_local()
                import torch
                text = self.tokenizer.apply_chat_template(messages, tokenize=False,
                    add_generation_prompt=True, enable_thinking=False)
                inputs = self.tokenizer(text, return_tensors='pt').to(self.model.device)
                if inputs['input_ids'].shape[1] > 2048:
                    raise ModelAccessError('Для учебного local-режима сократите вход до 2048 токенов.')
                with torch.inference_mode():
                    output = self.model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False)
                answer = self.tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            if not isinstance(answer, str) or not answer.strip():
                raise ModelAccessError('Нет текстового ответа; возможно, исчерпан лимит генерации.')
            status = 'ok'
            return answer
        finally:
            self.records.append({'backend': self.backend, 'model': self.model_id,
                                 'seconds': round(time.monotonic() - started, 3), 'status': status})

    def probe(self):
        try:
            answer = self.chat([{'role': 'user', 'content': 'Reply with the single word OK.'}], max_tokens=64)
            return {'status': 'MOCK_ONLY' if self.backend == 'mock' else 'TEXT_RESPONSE_RECEIVED',
                    'exact_ok': answer.strip() == 'OK', 'answer': answer,
                    'note': 'Не проверяет качество агента или native tool calling.'}
        except ModelAccessError as exc:
            return {'status': 'FAILED', 'reason': str(exc)}


In [4]:
import platform
print("Python:", platform.python_version())
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет; local будет медленнее")
except ImportError:
    print("torch не установлен; OpenRouter и mock доступны без него")
if BACKEND == "openrouter":
    catalog = free_models()
    for m in catalog:
        print(m["id"], "context:", m.get("context_length"), "tools (заявлено):", "tools" in (m.get("supported_parameters") or []))
    print("Наличие в каталоге не гарантирует успешный запрос. Выберите MODEL_ID выше.")

Python: 3.9.13
GPU: нет; local будет медленнее


In [5]:
if BACKEND == "openrouter" and not MODEL_ID:
    raise ValueError("Скопируйте id из каталога в MODEL_ID и повторите ячейку конфигурации.")
client = None
try:
    client = ModelClient(BACKEND, MODEL_ID, max_calls=MAX_CALLS)
    print(client.probe())
except ModelAccessError as exc:
    print({"status": "FAILED", "reason": str(exc)})
# При неудачной диагностике продолжайте обязательные задачи без модели.
# Для интеграции выберите mock и повторите конфигурацию и эту ячейку.

{'status': 'MOCK_ONLY', 'exact_ok': False, 'answer': '[MOCK: это заглушка, не ответ LLM]', 'note': 'Не проверяет качество агента или native tool calling.'}


`TEXT_RESPONSE_RECEIVED` означает только успешный текстовый запрос сейчас. `exact_ok=False` — нарушение запрошенного формата. `MOCK_ONLY` означает запуск заглушки. Поле `tools` в каталоге — заявление API, а не проверка вызова инструментов.

Диагностический запрос тоже расходует квоту. При 429 подождите или используйте local/mock; не перебирайте модели ради обхода лимита. Даже после успешной диагностики следующий запрос может завершиться ошибкой.